# SemEval-2026 Task 13 — Subtask A: GraphCodeBERT (v3 — Free-Tier Optimised)

**Binary Classification** — Detect machine-generated vs human-written code.

### Model: `microsoft/graphcodebert-base`
- Pre-trained with data flow information → structure-aware representations
- DFG-aware pre-training benefits persist without explicit DFG injection at fine-tune time

### v3 Changes from v2

| # | Change | Why |
|---|--------|-----|
| FIX 1 | LOLO holdout language → `javascript` (was `c++`) | `c++` is a *seen* language; HPO should optimise for unseen-language generalisation |
| FIX 2 | Focal loss & label smoothing decoupled | Mixing them broke `pt` estimation, defeating focal modulation |
| FIX 3 | CLS token → mean pooling | Mean pooling captures the full snippet; CLS has no "sentence start" semantic in code |
| FIX 4 | Stratified sampling by language × domain × label | Ensures seen-language/domain diversity even at 30 % data budget |
| FIX 5 | HPO trials 8 → 5; Phase 2 samples 100K → 150K | Redeploys saved HPO compute into more Phase 2 training data |
| FIX 6 | Middle-truncation for long snippets | Preserves signature (head) and return/logic (tail); discards boilerplate middle |
| FIX 7 | Warmup ratio 15 % → 6 %; `group_by_length=False` | 15 % was too aggressive; length-grouped batches introduced subtle batch bias |

### v3 Strategy: Two-Phase Approach
1. **Phase 1 — Optuna HPO Sweep** (~1.5 hrs, 5 trials on 20K subset)
2. **Phase 2 — Full Training** (~4 hrs, 150K stratified sample, 4 epochs)


In [1]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate pyarrow fastparquet tqdm optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
os.environ["WANDB_DISABLED"] = "true"

import gc
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import optuna
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaConfig,
    PreTrainedModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

2026-04-06 07:32:16.117761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775460736.277736      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775460736.330275      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775460736.739400      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775460736.739437      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775460736.739441      24 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Configuration

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Model ---
MODEL_NAME = "microsoft/graphcodebert-base"

# --- Data paths (Kaggle) ---
TRAIN_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
)
VAL_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
)
TEST_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
)

RANDOM_SEED = 42

# --- Validation Mode ---
# LOLO is not viable: unseen languages (Go, JS, PHP...) don't exist in 
# the training parquet, so HOLDOUT_LANGUAGE on those strings silently 
# falls back to stratified split. Java works mechanically but it's a 
# seen language — misleading signal and wastes 33% of training data.
USE_LOLO_VALIDATION        = False   # was True
UNSEEN_LANG_VAL_ONLY       = True    # NEW: filter official val set to unseen languages

# --- Fixed params (not tuned) ---
MAX_LENGTH       = 256
BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 2          # Effective batch = 32

# --- Optuna HPO sweep settings ---
# FIX 5: Reduced to 5 trials to save GPU quota for a larger Phase 2.
HPO_N_TRIALS       = 5        # was 8
HPO_SAMPLE_SIZE    = 20000
HPO_VAL_SIZE       = 5000
HPO_EPOCHS         = 2

# --- Full training (Phase 2) ---
# FIX 5 (cont.): Increased sample budget with saved HPO compute.
FULL_SAMPLE_SIZE    = 150000   # was 100000
FULL_VAL_SIZE       = 25000
UNSEEN_HOLDOUT_SIZE = 10000
FULL_EPOCHS         = 4        # was 5 — fits ~4 hrs on P100 with 150K
EARLY_STOPPING_PATIENCE = 2

# --- Output ---
OUTPUT_DIR = "/kaggle/working/results_graphcodebert_taskA_v3"

print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LENGTH}")
print(f"LOLO Validation: {USE_LOLO_VALIDATION}")
print(f"Unseen Language Val Only: {UNSEEN_LANG_VAL_ONLY}")
print(f"\n--- Phase 1: Optuna HPO ---")
print(f"Trials: {HPO_N_TRIALS}, Samples: {HPO_SAMPLE_SIZE}, Epochs/trial: {HPO_EPOCHS}")
print(f"\n--- Phase 2: Full Training ---")
print(f"Samples: {FULL_SAMPLE_SIZE}, Epochs: {FULL_EPOCHS}")
print(f"Unseen holdout: {UNSEEN_HOLDOUT_SIZE}")

Model: microsoft/graphcodebert-base
Max length: 256
LOLO Validation: True (Holdout: c++)

--- Phase 1: Optuna HPO ---
Trials: 8, Samples: 20000, Epochs/trial: 2

--- Phase 2: Full Training ---
Samples: 100000, Epochs: 5
Unseen holdout: 10000


## Model & Trainer Definitions

In [ ]:
# ============================================================
# Cell 4: Model, Trainer, & Data Utilities
# ============================================================

# ------------------------------------------------------------------
# Custom Model: Multi-Sample Dropout  +  Mean Pooling
# ------------------------------------------------------------------
class GraphCodeBERTMultiDropModel(PreTrainedModel):
    """
    GraphCodeBERT + multi-sample dropout head.

    FIX 3: Uses mean pooling over non-padding tokens instead of the bare
    [CLS] token.  For code there is no special "sentence start" semantic,
    so averaging all token representations captures more signal and
    consistently outperforms CLS-only in code classification tasks.
    """
    config_class = RobertaConfig
    supports_gradient_checkpointing = True

    @classmethod
    def _can_set_experts_implementation(cls):
        return False

    def __init__(self, config, num_labels=2, num_dropouts=5, head_dropout_base=0.15):
        super().__init__(config)
        self.num_labels = num_labels
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropouts = nn.ModuleList(
            [nn.Dropout(p=head_dropout_base + (i * 0.05)) for i in range(num_dropouts)]
        )
        self.out_proj = nn.Linear(config.hidden_size, num_labels)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask, **kwargs)

        # FIX 3: Mean pooling — average over real (non-padding) token positions.
        # Shape: (batch, seq_len, hidden) → (batch, hidden)
        token_embeddings = outputs[0]                          # (B, L, H)
        mask_expanded = attention_mask.unsqueeze(-1).float()   # (B, L, 1)
        pooled = (token_embeddings * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

        pooled = torch.tanh(self.dense(pooled))

        logits = torch.mean(
            torch.stack([self.out_proj(d(pooled)) for d in self.dropouts]),
            dim=0,
        )

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss, logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


# ------------------------------------------------------------------
# Custom Trainer: Focal Loss  (label smoothing removed — see FIX 2)
# ------------------------------------------------------------------
class FocalLossTrainer(Trainer):
    """
    FIX 2: Removed label_smoothing from inside focal loss computation.

    The original code passed label_smoothing into F.cross_entropy and then
    computed pt = exp(-ce_loss).  But label smoothing inflates the raw CE
    value (the soft targets are never fully satisfied), so pt is artificially
    depressed and the focal modulation (1-pt)^gamma over-weights *every*
    sample — effectively breaking the easy/hard distinction that focal loss
    is meant to provide.

    Fix: compute pt from clean (un-smoothed) probabilities, then optionally
    apply label smoothing as a *separate additive* regulariser on the final
    loss.  Controlled by `label_smoothing` kwarg (default 0 = off).
    """
    def __init__(self, class_weights=None, alpha=1.0, gamma=2.0,
                 label_smoothing=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = (
            class_weights.to(self.args.device) if class_weights is not None else None
        )
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing   # kept for API compat; see below

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if labels is not None:
            # Step 1: clean CE (no smoothing) → correct pt
            ce_loss_clean = F.cross_entropy(
                logits, labels.view(-1),
                weight=self.class_weights,
                reduction="none",
                label_smoothing=0.0,     # <-- always 0 here
            )
            pt = torch.exp(-ce_loss_clean)
            focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss_clean).mean()

            # Step 2 (optional): add smoothed CE as a *separate* regulariser
            # Only kicks in when label_smoothing > 0.
            if self.label_smoothing > 0.0:
                smooth_ce = F.cross_entropy(
                    logits, labels.view(-1),
                    weight=self.class_weights,
                    reduction="mean",
                    label_smoothing=self.label_smoothing,
                )
                focal_loss = focal_loss + self.label_smoothing * smooth_ce
        else:
            focal_loss = outputs.get("loss")

        return (focal_loss, outputs) if return_outputs else focal_loss


# ------------------------------------------------------------------
# Middle-truncation helper
# ------------------------------------------------------------------
def truncate_middle(code: str, tokenizer, max_length: int = MAX_LENGTH) -> str:
    """
    FIX 6: Keep the head and tail of a code snippet instead of blindly
    truncating from the right.

    Why: the head carries function signatures, imports, and class definitions;
    the tail carries return statements and final logic — both are more
    discriminative than the middle.  Long production/research snippets are
    the exact cases where this matters (test settings iii and iv).
    """
    ids = tokenizer.encode(code, add_special_tokens=False)
    if len(ids) <= max_length - 2:   # fits within budget (reserve 2 for CLS/SEP)
        return code
    keep = (max_length - 2) // 2
    kept_ids = ids[:keep] + ids[-keep:]
    return tokenizer.decode(kept_ids, skip_special_tokens=True,
                            clean_up_tokenization_spaces=True)


# ------------------------------------------------------------------
# Data loading utility
# ------------------------------------------------------------------
def load_data(sample_size, val_sample_size, holdout_size=0,
              random_seed=RANDOM_SEED, tokenizer=None):
    """
    Load data with two improvements:

    FIX 1 (LOLO): HOLDOUT_LANGUAGE is now an unseen language (javascript),
    so the HPO/validation signal reflects real cross-language generalisation.

    FIX 4 (Stratified sampling): When sub-sampling the training set we
    stratify by (language × domain × label) instead of label-only.  This
    ensures all seen language/domain combinations are represented even when
    using only 20–30 % of the data, giving the encoder broader coverage.
    """
    use_lolo = USE_LOLO_VALIDATION

    df = pd.read_parquet(TRAIN_PATH)
    df = df.dropna(subset=['code', 'label'])
    df['label'] = df['label'].astype(int)

    # Reserve unseen holdout *before* any sampling
    holdout_df = None
    if holdout_size > 0:
        holdout_df = df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(
                n=max(1, int(holdout_size * len(x) / len(df))),
                random_state=random_seed + 999,
            )
        )
        df = df.drop(holdout_df.index).reset_index(drop=True)
        holdout_df = holdout_df.reset_index(drop=True)
        print(f"  Unseen holdout reserved: {len(holdout_df)} samples")

    if use_lolo and 'language' in df.columns:
        # Fallback branch from earlier version that we won't use 
        df['language_lower'] = df['language'].str.lower()
        train_df = df[df['language_lower'] != "java"]
        val_df   = df[df['language_lower'] == "java"]

        if len(val_df) == 0:
            print(f"Warning: 'java' not found in training data, "
                  f"falling back to stratified split.")
            use_lolo = False
        else:
            # FIX 4: Stratify training sample by language × domain × label
            if sample_size and sample_size < len(train_df):
                strat_cols = [c for c in ['language_lower', 'domain', 'label']
                              if c in train_df.columns]
                n_per_stratum = max(1, sample_size // train_df[strat_cols].drop_duplicates().shape[0])
                train_df = (
                    train_df.groupby(strat_cols, group_keys=False)
                    .apply(lambda x: x.sample(
                        n=min(len(x), n_per_stratum),
                        random_state=random_seed,
                    ))
                    .reset_index(drop=True)
                )
                # Trim to exact budget if over
                if len(train_df) > sample_size:
                    train_df = train_df.sample(sample_size, random_state=random_seed)

            if val_sample_size and val_sample_size < len(val_df):
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(val_sample_size * len(x) / len(val_df))),
                        random_state=random_seed,
                    )
                )

            r = (train_df.reset_index(drop=True), val_df.reset_index(drop=True))
            return (*r, holdout_df) if holdout_df is not None else r

    # Fallback: stratified split (label-only, since no language column or LOLO off)
    if sample_size and sample_size < len(df):
        df = df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(
                n=max(1, int(sample_size * len(x) / len(df))),
                random_state=random_seed,
            )
        ).reset_index(drop=True)

    val_df = pd.read_parquet(VAL_PATH)
    val_df = val_df.dropna(subset=['code', 'label'])
    val_df['label'] = val_df['label'].astype(int)
    
    UNSEEN_LANGS_SET = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}

    if UNSEEN_LANG_VAL_ONLY and 'language' in val_df.columns:
        val_unseen = val_df[val_df['language'].str.lower().isin(UNSEEN_LANGS_SET)]
        if len(val_unseen) >= 500:   # sanity check: enough samples to be meaningful
            val_df = val_unseen
            print(f"  Val filtered to unseen languages: {len(val_df)} samples")
            print(f"  Languages: {sorted(val_df['language'].str.lower().unique())}")
        else:
            print("  Warning: too few unseen-language val samples, using full val set")

    if val_sample_size and val_sample_size < len(val_df):
        val_df = val_df.groupby('label', group_keys=False).apply(
            lambda x: x.sample(
                n=max(1, int(val_sample_size * len(x) / len(val_df))),
                random_state=random_seed,
            )
        ).reset_index(drop=True)

    r = (df, val_df)
    return (*r, holdout_df) if holdout_df is not None else r


def tokenize_and_prepare(train_df, val_df, tokenizer, max_length=MAX_LENGTH):
    """
    Tokenize dataframes into HF Datasets.
    FIX 6: Applies middle-truncation *before* tokenisation for long snippets.
    """
    # Apply middle truncation to raw text first
    print("  Applying middle-truncation to long snippets ...")
    train_df = train_df.copy()
    val_df   = val_df.copy()
    train_df['code'] = train_df['code'].apply(
        lambda c: truncate_middle(c, tokenizer, max_length)
    )
    val_df['code'] = val_df['code'].apply(
        lambda c: truncate_middle(c, tokenizer, max_length)
    )

    def tok_fn(examples):
        return tokenizer(
            examples['code'], truncation=True, padding=True,
            max_length=max_length,
        )

    train_ds = Dataset.from_pandas(train_df[['code', 'label']])
    val_ds   = Dataset.from_pandas(val_df[['code', 'label']])
    train_ds = train_ds.map(tok_fn, batched=True, remove_columns=['code'])
    val_ds   = val_ds.map(tok_fn, batched=True, remove_columns=['code'])
    train_ds = train_ds.rename_column('label', 'labels')
    val_ds   = val_ds.rename_column('label', 'labels')
    return train_ds, val_ds


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
    }


def build_model(hidden_dropout, attention_dropout, freeze_layers, head_dropout_base=0.15):
    """Build model with specified hyperparameters."""
    config = RobertaConfig.from_pretrained(MODEL_NAME)
    config.hidden_dropout_prob = hidden_dropout
    config.attention_probs_dropout_prob = attention_dropout

    model = GraphCodeBERTMultiDropModel.from_pretrained(
        MODEL_NAME, config=config, num_labels=2,
        head_dropout_base=head_dropout_base,
    ).to('cuda')

    if freeze_layers > 0:
        for param in model.roberta.embeddings.parameters():
            param.requires_grad = False
        for i in range(freeze_layers):
            for param in model.roberta.encoder.layer[i].parameters():
                param.requires_grad = False

    return model


print("All definitions loaded.")



All definitions loaded.


---
## Phase 1: Optuna Hyperparameter Search

Search space:
| Parameter | Range | Type |
|-----------|-------|------|
| `learning_rate` | 5e-6 — 3e-5 | log-uniform |
| `weight_decay` | 0.01 — 0.10 | uniform |
| `freeze_layers` | 0, 4, 6, 8 | categorical |
| `hidden_dropout` | 0.1 — 0.3 | uniform |
| `attention_dropout` | 0.1 — 0.25 | uniform |
| `label_smoothing` | 0.0 — 0.15 | uniform |
| `head_dropout_base` | 0.1 — 0.25 | uniform |

Each trial: 20K samples, 2 epochs (~15–20 min per trial).

In [5]:
# ============================================================
# Cell 5: Prepare HPO data (load once, reuse across trials)
# ============================================================

print("Loading HPO subset ...")
hpo_train_df, hpo_val_df = load_data(
    sample_size=HPO_SAMPLE_SIZE,
    val_sample_size=HPO_VAL_SIZE,
)
print(f"HPO Train: {len(hpo_train_df)}, HPO Val: {len(hpo_val_df)}")

# Tokenize once (tokenizer is the same across all trials)
print("Tokenizing HPO data ...")
hpo_tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
hpo_train_ds, hpo_val_ds = tokenize_and_prepare(
    hpo_train_df, hpo_val_df, hpo_tokenizer,
)
print(f"HPO datasets ready. Train: {len(hpo_train_ds)}, Val: {len(hpo_val_ds)}")

Loading HPO subset ...
HPO Train: 19999, HPO Val: 4999
Tokenizing HPO data ...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

Map:   0%|          | 0/19999 [00:00<?, ? examples/s]

Map:   0%|          | 0/4999 [00:00<?, ? examples/s]

HPO datasets ready. Train: 19999, Val: 4999


In [ ]:
# ============================================================
# Cell 6: Optuna HPO Sweep
# ============================================================
# FIX 5: Reduced to 5 trials.  We narrow the search to the 3 highest-impact
# hyperparameters (lr, freeze, head_dropout) and fix the rest to sensible
# defaults, so 5 trials gives reasonable coverage without burning GPU quota.
#
# FIX 2: label_smoothing is now sampled and passed to FocalLossTrainer where
# it is applied correctly as a *separate* regulariser, not mixed into pt.

# --- PRECOMPUTE CLASS WEIGHTS ON HPO SUBSET ---
_labels = hpo_train_df['label'].values
_counts = np.bincount(_labels)
_hpo_class_weights = torch.tensor([len(_labels) / c for c in _counts], dtype=torch.float32)
# Normalize to keep loss scale somewhat stable
_hpo_class_weights = _hpo_class_weights / _hpo_class_weights.sum()
print(f"HPO Class Weights: {_hpo_class_weights.tolist()}")


def objective(trial):
    """Optuna objective: train with sampled HPs, return val macro_f1."""

    # High-impact params — search these
    lr         = trial.suggest_float("learning_rate", 1e-6, 2e-5, log=True) # Dropped upper bound to prevent collapse
    freeze     = trial.suggest_categorical("freeze_layers", [0, 4, 6, 8])
    head_drop  = trial.suggest_float("head_dropout_base", 0.10, 0.25)

    # Lower-impact params — narrow ranges to reduce search dimensionality
    wd         = trial.suggest_float("weight_decay", 0.01, 0.05)
    h_drop     = trial.suggest_float("hidden_dropout", 0.10, 0.20)
    a_drop     = trial.suggest_float("attention_dropout", 0.10, 0.20)
    lbl_smooth = trial.suggest_float("label_smoothing", 0.0, 0.10)

    trial_dir = f"/kaggle/working/hpo_trial_{trial.number}"

    print(f"\n{'─'*60}")
    print(f"Trial {trial.number}: lr={lr:.2e}, freeze={freeze}, "
          f"head_drop={head_drop:.2f}, wd={wd:.3f}, "
          f"h_drop={h_drop:.2f}, a_drop={a_drop:.2f}, lbl_smooth={lbl_smooth:.2f}")
    print(f"{'─'*60}")

    try:
        model = build_model(
            hidden_dropout=h_drop,
            attention_dropout=a_drop,
            freeze_layers=freeze,
            head_dropout_base=head_drop,
        )

        steps_per_epoch = len(hpo_train_ds) // (BATCH_SIZE * GRAD_ACCUM_STEPS)

        training_args = TrainingArguments(
            output_dir=trial_dir,
            num_train_epochs=HPO_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            weight_decay=wd,
            # FIX: warmup = 6% of total steps (was 15% — too aggressive)
            warmup_steps=int(steps_per_epoch * HPO_EPOCHS * 0.06),
            logging_steps=200,
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            remove_unused_columns=False,
            learning_rate=lr,
            lr_scheduler_type="cosine",
            report_to=[],
            fp16=True,
            fp16_full_eval=True,
            gradient_checkpointing=True,
            max_grad_norm=1.0,           # Anti-collapse: strict gradient clipping
            group_by_length=False,       # FIX: disabled to avoid length-correlated batches
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            optim="adamw_torch_fused",
        )

        data_collator = DataCollatorWithPadding(tokenizer=hpo_tokenizer)

        trainer = FocalLossTrainer(
            model=model,
            args=training_args,
            train_dataset=hpo_train_ds,
            eval_dataset=hpo_val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            alpha=1.0,
            gamma=3.0,                   # Adjusted focal gamma for imbalance
            label_smoothing=lbl_smooth,  # now applied correctly in FocalLossTrainer
            class_weights=_hpo_class_weights,  # Anti-collapse: penalty for missing minority
        )

        trainer.train()
        metrics  = trainer.evaluate()
        val_f1   = metrics.get("eval_macro_f1", 0.0)

    except Exception as e:
        print(f"Trial {trial.number} FAILED: {e}")
        val_f1 = 0.0

    finally:
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()
        import shutil
        if os.path.exists(trial_dir):
            shutil.rmtree(trial_dir, ignore_errors=True)

    return val_f1


# --- Run the sweep ---
print(f"\n{'='*70}")
print(f"  PHASE 1: Optuna HPO Sweep ({HPO_N_TRIALS} trials)")
print(f"{'='*70}")

sweep_start = time.time()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
)
study.optimize(objective, n_trials=HPO_N_TRIALS)

sweep_elapsed = time.time() - sweep_start
print(f"\nHPO Sweep completed in {sweep_elapsed/3600:.1f} hours")


  PHASE 1: Optuna HPO Sweep (8 trials)

────────────────────────────────────────────────────────────
Trial 0: lr=9.78e-06, wd=0.096, freeze=0, h_drop=0.11, a_drop=0.23, lbl_smooth=0.09, head_drop=0.21
────────────────────────────────────────────────────────────


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of GraphCodeBERTMultiDropModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['dense.bias', 'dense.weight', 'out_proj.bias', 'out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



────────────────────────────────────────────────────────────
Trial 1: lr=5.19e-06, wd=0.097, freeze=0, h_drop=0.16, a_drop=0.18, lbl_smooth=0.06, head_drop=0.14
────────────────────────────────────────────────────────────


Some weights of GraphCodeBERTMultiDropModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['dense.bias', 'dense.weight', 'out_proj.bias', 'out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...


────────────────────────────────────────────────────────────
Trial 6: lr=1.00e-05, wd=0.034, freeze=0, h_drop=0.13, a_drop=0.22, lbl_smooth=0.01, head_drop=0.25
────────────────────────────────────────────────────────────


Some weights of GraphCodeBERTMultiDropModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['dense.bias', 'dense.weight', 'out_proj.bias', 'out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



────────────────────────────────────────────────────────────
Trial 7: lr=1.99e-05, wd=0.028, freeze=4, h_drop=0.25, a_drop=0.11, lbl_smooth=0.05, head_drop=0.12
────────────────────────────────────────────────────────────


Some weights of GraphCodeBERTMultiDropModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['dense.bias', 'dense.weight', 'out_proj.bias', 'out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



HPO Sweep completed in 3.6 hours


In [7]:
# ============================================================
# Cell 7: HPO Results Summary
# ============================================================

print(f"\n{'='*70}")
print(f"  OPTUNA RESULTS")
print(f"{'='*70}")

# All trials summary
print("\n--- All Trials ---")
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
display_cols = ["number", "value", "params_learning_rate", "params_weight_decay",
                "params_freeze_layers", "params_hidden_dropout",
                "params_attention_dropout", "params_label_smoothing",
                "params_head_dropout_base", "duration"]
print(trials_df[[c for c in display_cols if c in trials_df.columns]].to_string(index=False))

# Best trial
best = study.best_trial
print(f"\n--- Best Trial: #{best.number} ---")
for k, v in best.params.items():
    print(f"  {k}: {v}")

# Store best params for Phase 2
BEST_LR             = best.params["learning_rate"]
BEST_WD             = best.params["weight_decay"]
BEST_FREEZE         = best.params["freeze_layers"]
BEST_H_DROP         = best.params["hidden_dropout"]
BEST_A_DROP         = best.params["attention_dropout"]
BEST_LBL_SMOOTH     = best.params["label_smoothing"]
BEST_HEAD_DROP      = best.params["head_dropout_base"]

# Free HPO data
del hpo_train_ds, hpo_val_ds, hpo_train_df, hpo_val_df
gc.collect()
torch.cuda.empty_cache()
print("\nHPO data freed. Ready for Phase 2.")


  OPTUNA RESULTS

--- All Trials ---
 number   value  params_learning_rate  params_weight_decay  params_freeze_layers  params_hidden_dropout  params_attention_dropout  params_label_smoothing  params_head_dropout_base               duration
      0 0.32272              0.000010             0.095564                     0               0.111617                  0.229926                0.090167                  0.206211 0 days 00:46:21.119864
      1 0.32272              0.000005             0.097292                     0               0.160848                  0.178713                0.064792                  0.143684 0 days 00:46:15.250381
      2 0.32272              0.000015             0.022554                     8               0.139935                  0.177135                0.088862                  0.106968 0 days 00:08:05.615673
      3 0.32272              0.000015             0.025347                     6               0.160923                  0.114651                0.102

---
## Phase 2: Full Training with Best Hyperparameters

Uses the Optuna-selected hyperparameters on the full 100K sample.

In [ ]:
# ============================================================
# Cell 8: Full training with best HPs
# ============================================================

print(f"\n{'='*70}")
print(f"  PHASE 2: Full Training with Optuna-Best Hyperparameters")
print(f"{'='*70}")
print(f"  LR:             {BEST_LR:.2e}")
print(f"  Weight Decay:   {BEST_WD:.4f}")
print(f"  Freeze Layers:  {BEST_FREEZE}")
print(f"  Hidden Dropout: {BEST_H_DROP:.3f}")
print(f"  Attn Dropout:   {BEST_A_DROP:.3f}")
print(f"  Label Smooth:   {BEST_LBL_SMOOTH:.3f}")
print(f"  Head Dropout:   {BEST_HEAD_DROP:.3f}")
print(f"  Samples: {FULL_SAMPLE_SIZE}, Epochs: {FULL_EPOCHS}")
print()

# --- Load full data ---
print("Loading full training data ...")
full_train_df, full_val_df, unseen_holdout_df = load_data(
    sample_size=FULL_SAMPLE_SIZE,
    val_sample_size=FULL_VAL_SIZE,
    holdout_size=UNSEEN_HOLDOUT_SIZE,
)
print(f"Full Train: {len(full_train_df)}, Val: {len(full_val_df)}, Holdout: {len(unseen_holdout_df)}")


# --- COMPUTE CLASS WEIGHTS ON FULL SUBSET ---
_f_labels = full_train_df['label'].values
_f_counts = np.bincount(_f_labels)
_full_class_weights = torch.tensor([len(_f_labels) / c for c in _f_counts], dtype=torch.float32)
_full_class_weights = _full_class_weights / _full_class_weights.sum()
print(f"Full Train Class Weights: {_full_class_weights.tolist()}")


# --- Tokenize ---
print("Tokenizing ...")
full_tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
full_train_ds, full_val_ds = tokenize_and_prepare(
    full_train_df, full_val_df, full_tokenizer,
)

# --- Build model with best HPs ---
print("Building model with best hyperparameters ...")
full_model = build_model(
    hidden_dropout=BEST_H_DROP,
    attention_dropout=BEST_A_DROP,
    freeze_layers=BEST_FREEZE,
    head_dropout_base=BEST_HEAD_DROP,
)

trainable = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in full_model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

# --- Training ---
steps_per_epoch = len(full_train_ds) // (BATCH_SIZE * GRAD_ACCUM_STEPS)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=FULL_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    weight_decay=BEST_WD,
    warmup_steps=int(steps_per_epoch * FULL_EPOCHS * 0.06),  # FIX: was 0.15 (too high)
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    remove_unused_columns=False,
    learning_rate=BEST_LR,
    lr_scheduler_type="cosine",
    save_total_limit=2,
    report_to=[],
    fp16=True,
    fp16_full_eval=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,           # Anti-collapse: strict gradient clipping
    group_by_length=False,       # FIX: disabled to avoid length-correlated batches
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    optim="adamw_torch_fused",
)

data_collator = DataCollatorWithPadding(tokenizer=full_tokenizer)

full_trainer = FocalLossTrainer(
    model=full_model,
    args=training_args,
    train_dataset=full_train_ds,
    eval_dataset=full_val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
    )],
    alpha=1.0,
    gamma=3.0,                   # Adjusted focal gamma
    label_smoothing=BEST_LBL_SMOOTH,
    class_weights=_full_class_weights,  # Pass computed class weights
)

train_start = time.time()
full_trainer.train()
train_elapsed = time.time() - train_start
print(f"\nFull training completed in {train_elapsed/3600:.1f} hours")

# Save
full_trainer.save_model()
full_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")


  PHASE 2: Full Training with Optuna-Best Hyperparameters
  LR:             9.78e-06
  Weight Decay:   0.0956
  Freeze Layers:  0
  Hidden Dropout: 0.112
  Attn Dropout:   0.230
  Label Smooth:   0.090
  Head Dropout:   0.206
  Samples: 100000, Epochs: 5

Loading full training data ...
  Unseen holdout reserved: 9999 samples
Full Train: 99999, Val: 22918, Holdout: 9999
Tokenizing ...


Map:   0%|          | 0/99999 [00:00<?, ? examples/s]

Map:   0%|          | 0/22918 [00:00<?, ? examples/s]

Building model with best hyperparameters ...


Some weights of GraphCodeBERTMultiDropModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['dense.bias', 'dense.weight', 'out_proj.bias', 'out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable: 124,647,170 / 124,647,170 params



Full training completed in 6.0 hours
Model saved to /kaggle/working/results_graphcodebert_taskA_v2


In [9]:
# ============================================================
# Cell 9: Validation Evaluation
# ============================================================

print(f"\n{'='*60}")
print(f"  VALIDATION EVALUATION — GraphCodeBERT v2")
print(f"{'='*60}")

predictions = full_trainer.predict(full_val_ds)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("\nPer-Class Classification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["human", "machine"],
    zero_division=0,
))

macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)



  VALIDATION EVALUATION — GraphCodeBERT v2



Per-Class Classification Report:
              precision    recall  f1-score   support

       human       0.48      1.00      0.65     10917
     machine       0.00      0.00      0.00     12001

    accuracy                           0.48     22918
   macro avg       0.24      0.50      0.32     22918
weighted avg       0.23      0.48      0.31     22918




## Unseen Holdout Evaluation

10K labeled samples reserved from the training pool **before** any HPO or training. The model never saw these — truest generalization measure since test set has no labels.

In [10]:
# ============================================================
# Unseen Holdout Evaluation — 10K labeled samples never seen
# ============================================================
from tqdm import tqdm
print(f"\n{"="*70}")
print(f"  UNSEEN HOLDOUT EVALUATION")
print(f"{"="*70}")
print(f"  Size: {len(unseen_holdout_df)}, Labels: {unseen_holdout_df['label'].value_counts().to_dict()}")
full_model.eval()
holdout_preds = []
with torch.no_grad():
    for i in tqdm(range(0, len(unseen_holdout_df), BATCH_SIZE*2), desc="Holdout"):
        batch = unseen_holdout_df["code"].tolist()[i:i+BATCH_SIZE*2]
        enc = full_tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
        logits = full_model(input_ids=enc["input_ids"].to("cuda"), attention_mask=enc["attention_mask"].to("cuda")).logits
        holdout_preds.extend(logits.argmax(dim=-1).cpu().tolist())
y_true_h = unseen_holdout_df["label"].values
y_pred_h = np.array(holdout_preds)
print("\nClassification Report (Unseen Holdout):")
print(classification_report(y_true_h, y_pred_h, target_names=["human","machine"], zero_division=0))
holdout_f1 = f1_score(y_true_h, y_pred_h, average="macro", zero_division=0)
holdout_acc = accuracy_score(y_true_h, y_pred_h)
print(f"\n--- Generalization Gap ---")
print(f"  Gap:              {macro_f1 - holdout_f1:+.4f}")
if "language" in unseen_holdout_df.columns:
    print("\n--- Per-Language Breakdown (Holdout) ---")
    for lang in sorted(unseen_holdout_df["language"].str.lower().unique()):
        mask = unseen_holdout_df["language"].str.lower() == lang
        if mask.sum() < 10: continue
        lf1 = f1_score(y_true_h[mask], y_pred_h[mask], average="macro", zero_division=0)
        lacc = accuracy_score(y_true_h[mask], y_pred_h[mask])
        print(f"  {lang:>12s} (n={mask.sum():>5d}): Acc={lacc:.4f} F1={lf1:.4f}")



  UNSEEN HOLDOUT EVALUATION
  Size: 9999, Labels: {1: 5230, 0: 4769}


Holdout: 100%|██████████| 313/313 [00:37<00:00,  8.36it/s]



Classification Report (Unseen Holdout):
              precision    recall  f1-score   support

       human       0.48      1.00      0.65      4769
     machine       0.00      0.00      0.00      5230

    accuracy                           0.48      9999
   macro avg       0.24      0.50      0.32      9999
weighted avg       0.23      0.48      0.31      9999


--- Generalization Gap ---
  Gap:              -0.0003

--- Per-Language Breakdown (Holdout) ---
           c++ (n=  474): Acc=0.4852 F1=0.3267
          java (n=  364): Acc=0.4396 F1=0.3053
        python (n= 9161): Acc=0.4780 F1=0.3234


## Test Set Evaluation

In [11]:
# ============================================================
# Cell 10: Test-set evaluation utilities
# ============================================================
from tqdm import tqdm

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}


@torch.no_grad()
def evaluate_on_test(model, tokenizer, parquet_path, max_length=MAX_LENGTH,
                     batch_size=32, device=None):
    """Run inference on test parquet."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    preds = []
    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = preds
    return df


def evaluate_by_category(df, tag="GraphCodeBERT-v2"):
    """Print classification metrics for 4 evaluation settings + overall."""
    lang_col = next(
        (c for c in df.columns if c.lower() in ('language', 'lang', 'programming_language')),
        None,
    )
    domain_col = next(
        (c for c in df.columns if c.lower() in ('domain', 'task_type', 'category')),
        None,
    )

    if 'label' not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    y_true, y_pred = df['label'].values, df['prediction'].values
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS  --  {tag}")
    print(f"{'='*70}")

    print("\nOverall Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    if lang_col is not None and domain_col is not None:
        _norm = lambda v: str(v).strip().lower()
        df['_l'] = df[lang_col].apply(_norm)
        df['_d'] = df[domain_col].apply(_norm)

        settings = [
            ("(i)   Seen Lang & Seen Domain",      SEEN_LANGS,   SEEN_DOMAINS),
            ("(ii)  Unseen Lang & Seen Domain",    UNSEEN_LANGS, SEEN_DOMAINS),
            ("(iii) Seen Lang & Unseen Domain",    SEEN_LANGS,   UNSEEN_DOMAINS),
            ("(iv)  Unseen Lang & Unseen Domain",  UNSEEN_LANGS, UNSEEN_DOMAINS),
        ]

        for name, langs, domains in settings:
            mask = df['_l'].isin(langs) & df['_d'].isin(domains)
            sub = df[mask]
            n = len(sub)
            if n == 0:
                print(f"\n  {name}:  ** no samples **")
                continue
            y_t, y_p = sub['label'].values, sub['prediction'].values
            sub_acc = accuracy_score(y_t, y_p)
            sub_mf1 = f1_score(y_t, y_p, average='macro', zero_division=0)
            print(f"\n  {name}  (n={n})")
            print(classification_report(y_t, y_p, zero_division=0))

        df.drop(columns=['_l', '_d'], inplace=True)

    print("=" * 70)

In [12]:
# ============================================================
# Cell 11: Run test evaluation
# ============================================================

if os.path.exists(TEST_PATH):
    test_df = evaluate_on_test(
        model=full_model,
        tokenizer=full_tokenizer,
        parquet_path=TEST_PATH,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE * 2,
        device="cuda",
    )
    evaluate_by_category(test_df, tag="GraphCodeBERT-TaskA-v2")
else:
    print(f"Test file not found at {TEST_PATH}. Skipping test evaluation.")

Predicting: 100%|██████████| 32/32 [00:04<00:00,  6.70it/s]



  TEST RESULTS  --  GraphCodeBERT-TaskA-v2

Overall Classification Report:
              precision    recall  f1-score   support

           0       0.78      1.00      0.87       777
           1       0.00      0.00      0.00       223

    accuracy                           0.78      1000
   macro avg       0.39      0.50      0.44      1000
weighted avg       0.60      0.78      0.68      1000



In [13]:
# ============================================================
# Cell 12: Clean up
# ============================================================

del full_trainer, full_model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")

GPU memory freed.
